# Start here: capstone N-PORT data
Run `python run.py` first. This notebook reads the resulting database; it does not require a server.

The default universe has 59 series, including benchmarks. PRIMARY is the core 49-fund analysis group. Read README.md before using this data for prediction.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import research

DB = Path("output/nport.sqlite")
assert DB.exists(), "Run python run.py first, or set DB to your output database"
con = sqlite3.connect(DB)


## 1. Check coverage before analyzing results

In [2]:
coverage = pd.read_sql_query("SELECT * FROM coverage", con)
display(coverage.groupby("group").agg(funds=("series_id", "size"), observed=("report_count", lambda x: (x > 0).sum())))
display(coverage[["series_id", "fund_name", "report_count", "first_report_date", "last_report_date"]])


,funds,observed
group,,
ADJACENT,4,4
BENCHMARK,4,4
CHECK,2,2
PRIMARY,49,49


,series_id,fund_name,report_count,first_report_date,last_report_date
0,S000000713,Western Asset Core Bond Fund,28,2019-09-30,2026-06-30
1,S000000714,Western Asset Core Plus Bond Fund,28,2019-09-30,2026-06-30
2,S000000753,Baird Aggregate Bond Fund,28,2019-09-30,2026-06-30
3,S000000754,Baird Core Plus Bond Fund,28,2019-09-30,2026-06-30
4,S000001147,TCW MetWest Total Return Bond Fund,28,2019-09-30,2026-06-30
5,S000002136,T. Rowe Price New Income Fund,27,2019-11-29,2026-05-31
6,S000002497,MFS Total Return Bond Fund,28,2019-10-31,2026-07-31
7,S000002564,Vanguard Total Bond Market Index Fund,28,2019-09-30,2026-06-30
8,S000002586,Vanguard Interm-Term Investment-Grade Fund,28,2019-10-31,2026-07-31
9,S000003477,JPMorgan Core Bond Fund,27,2019-11-30,2026-05-31


## 2. Your main fund-quarter table
One row per exact report date, preserving different fiscal calendars. The normalized risk columns retain reported signs; validate the convention before interpreting economic duration.

In [3]:
fund_quarter = pd.read_sql_query("SELECT * FROM fund_quarter WHERE universe_group = 'PRIMARY'", con)
display(fund_quarter[["series_id", "fund_name", "report_date", "net_assets", "usd_rate_sensitivity_years_signed", "ig_spread_sensitivity_years_signed"]].head(15))
risk = pd.read_sql_query("SELECT * FROM risk_normalized_latest", con)
display(risk.head())


,series_id,fund_name,report_date,net_assets,usd_rate_sensitivity_years_signed,ig_spread_sensitivity_years_signed
0,S000000713,Western Asset Core Bond Fund,2019-09-30,1.199395e+10,6.192880,3.709030
1,S000000713,Western Asset Core Bond Fund,2019-12-31,1.347645e+10,6.462171,3.481142
2,S000000713,Western Asset Core Bond Fund,2020-03-31,1.460224e+10,5.444279,3.882038
3,S000000713,Western Asset Core Bond Fund,2020-06-30,1.640822e+10,7.089250,5.087204
4,S000000713,Western Asset Core Bond Fund,2020-09-30,1.863354e+10,7.266561,5.144694
5,S000000713,Western Asset Core Bond Fund,2020-12-31,2.003960e+10,7.396972,5.465622
6,S000000713,Western Asset Core Bond Fund,2021-03-31,1.996357e+10,7.352200,5.276334
7,S000000713,Western Asset Core Bond Fund,2021-06-30,2.072074e+10,7.358109,4.753510
8,S000000713,Western Asset Core Bond Fund,2021-09-30,2.146672e+10,7.085307,4.120961
9,S000000713,Western Asset Core Bond Fund,2021-12-31,2.093629e+10,7.331309,4.650445


,source_archive_id,accession_number,series_id,report_date,risk_row_id,risk_type,currency,credit_bucket,tenor,shock_bp,value_change_usd,source_column,filing_date,available_date_assumed,availability_basis,net_assets,reported_sensitivity_years
0,f8e10bce83ac2251bcdb80111faa362b7c2b1ffb658151...,0001752724-19-180372,S000000713,2019-09-30,fund,credit_spread,USD,INVEST,3MON,1,9695.76,CREDIT_SPREAD_3MON_INVEST,2019-11-27,2019-11-30,"max(filing_date, report_date+60d)+1d; unverifi...",1.199395e+10,0.008084
1,f8e10bce83ac2251bcdb80111faa362b7c2b1ffb658151...,0001752724-19-180372,S000000713,2019-09-30,fund,credit_spread,USD,INVEST,1YR,1,826794.46,CREDIT_SPREAD_1YR_INVEST,2019-11-27,2019-11-30,"max(filing_date, report_date+60d)+1d; unverifi...",1.199395e+10,0.689343
2,f8e10bce83ac2251bcdb80111faa362b7c2b1ffb658151...,0001752724-19-180372,S000000713,2019-09-30,fund,credit_spread,USD,INVEST,5YR,1,2491936.76,CREDIT_SPREAD_5YR_INVEST,2019-11-27,2019-11-30,"max(filing_date, report_date+60d)+1d; unverifi...",1.199395e+10,2.077661
3,f8e10bce83ac2251bcdb80111faa362b7c2b1ffb658151...,0001752724-19-180372,S000000713,2019-09-30,fund,credit_spread,USD,INVEST,10YR,1,1217996.48,CREDIT_SPREAD_10YR_INVEST,2019-11-27,2019-11-30,"max(filing_date, report_date+60d)+1d; unverifi...",1.199395e+10,1.015509
4,f8e10bce83ac2251bcdb80111faa362b7c2b1ffb658151...,0001752724-19-180372,S000000713,2019-09-30,fund,credit_spread,USD,INVEST,30YR,1,-97830.63,CREDIT_SPREAD_30YR_INVEST,2019-11-27,2019-11-30,"max(filing_date, report_date+60d)+1d; unverifi...",1.199395e+10,-0.081567


## 3. Returns and flows have different keys
Returns are by share class. Flows are by fund. Do not join flows to every class and then sum them.

In [4]:
returns = pd.read_sql_query("SELECT * FROM monthly_returns_latest", con)
flows = pd.read_sql_query("SELECT * FROM monthly_flows_latest", con)
class_coverage = returns.groupby(["series_id", "class_id"], dropna=False).agg(months=("month_end", "nunique"), first_month=("month_end", "min"), last_month=("month_end", "max")).reset_index()
display(class_coverage.head(20))


,series_id,class_id,months,first_month,last_month
0,S000000713,C000002079,84,2019-07-31,2026-06-30
1,S000000713,C000002080,84,2019-07-31,2026-06-30
2,S000000713,C000066988,84,2019-07-31,2026-06-30
3,S000000713,C000113701,84,2019-07-31,2026-06-30
4,S000000713,C000113702,84,2019-07-31,2026-06-30
5,S000000713,C000113703,84,2019-07-31,2026-06-30
6,S000000713,C000117034,84,2019-07-31,2026-06-30
7,S000000714,C000002081,84,2019-07-31,2026-06-30
8,S000000714,C000002082,84,2019-07-31,2026-06-30
9,S000000714,C000066989,84,2019-07-31,2026-06-30


### Optional reference-class selection
Create `reference_classes.csv` with columns `series_id,class_id`, using verified class identities. Only include the funds you have mapped. Use `__NO_CLASS__` only for genuinely classless reported returns. Do not choose the first class alphabetically. This simple helper supports one stable class per series; if a class changes historically, build a separate dated mapping.

In [5]:
mapping_path = Path("reference_classes.csv")
if mapping_path.exists():
    mapping = pd.read_csv(mapping_path, dtype=str)
    fund_month = research.reference_class_panel(mapping, DB)
    display(fund_month.head())
else:
    print("All class returns are ready. Add reference_classes.csv when class selection is verified.")


All class returns are ready. Add reference_classes.csv when class selection is verified.


## 4. Query one fund without loading the entire holdings history

In [6]:
series_id = "S000053068"  # Vanguard Core Bond Fund, from the supplied universe
latest_report = con.execute("SELECT MAX(report_date) FROM holdings_latest WHERE series_id=?", (series_id,)).fetchone()[0]
holdings = pd.read_sql_query("SELECT * FROM holdings_enriched_latest WHERE series_id=? AND report_date=?", con, params=(series_id, latest_report))
display(holdings.head())
print("Positions:", len(holdings), "Reported weight sum:", holdings.PERCENTAGE.sum(min_count=1))
# This is not a duration estimate. Derivative market values can be very small.


,series_id,report_date,report_month,filing_date,available_date_assumed,availability_basis,source_archive_id,source_row,ACCESSION_NUMBER,HOLDING_ID,...,FAIR_VALUE_LEVEL,DERIVATIVE_CAT,weight_fraction,is_reported_derivative,MATURITY_DATE,COUPON_TYPE,coupon_pct,coupon_decimal,IS_DEFAULT,years_to_maturity
0,S000053068,2026-06-30,2026-06-30,2026-08-28,2026-08-30,"max(filing_date, report_date+60d)+1d; unverifi...",2f5ee059234d2e28f16d0451d9fcb405aab3dad5d92cde...,4633113,0000836906-26-000476,177105226,...,2,None,3.730127e-04,0,2055-07-25,Fixed,4.9300,0.049300,N,29.067762
1,S000053068,2026-06-30,2026-06-30,2026-08-28,2026-08-30,"max(filing_date, report_date+60d)+1d; unverifi...",2f5ee059234d2e28f16d0451d9fcb405aab3dad5d92cde...,4633114,0000836906-26-000476,177106912,...,2,None,2.328256e-05,0,2031-03-15,Fixed,5.3500,0.053500,N,4.706366
2,S000053068,2026-06-30,2026-06-30,2026-08-28,2026-08-30,"max(filing_date, report_date+60d)+1d; unverifi...",2f5ee059234d2e28f16d0451d9fcb405aab3dad5d92cde...,4633115,0000836906-26-000476,177103538,...,2,None,3.374403e-07,0,2029-09-15,Fixed,3.3000,0.033000,N,3.211499
3,S000053068,2026-06-30,2026-06-30,2026-08-28,2026-08-30,"max(filing_date, report_date+60d)+1d; unverifi...",2f5ee059234d2e28f16d0451d9fcb405aab3dad5d92cde...,4633116,0000836906-26-000476,177103874,...,2,None,9.092770e-06,0,2045-06-01,Fixed,4.1500,0.041500,N,18.921287
4,S000053068,2026-06-30,2026-06-30,2026-08-28,2026-08-30,"max(filing_date, report_date+60d)+1d; unverifi...",2f5ee059234d2e28f16d0451d9fcb405aab3dad5d92cde...,4633117,0000836906-26-000476,177107886,...,2,None,1.145013e-04,0,2044-07-25,Floating,4.6278,0.046278,N,18.069815


Positions: 3537 Reported weight sum: 96.837165476368


## 5. Read derivative details without multiplying positions

In [7]:
swaps = pd.read_sql_query("SELECT * FROM nonforeign_exchange_swap_latest WHERE series_id=? AND report_date=?", con, params=(series_id, latest_report))
display(swaps.head())
# Includes different swap types. Inspect reference instruments/payment legs before labeling an interest-rate swap.
# Never infer net duration direction from pay-fixed vs receive-fixed NOTIONAL totals alone.


,series_id,report_date,ACCESSION_NUMBER,source_archive_id,source_row,HOLDING_ID,SWAP_FLAG,TERMINATION_DATE,UPFRONT_PAYMENT,PMNT_CURRENCY_CODE,...,FLOATING_RATE_INDEX_RECEIPT,FLOATING_RATE_SPREAD_RECEIPT,CURRENCY_CODE_RECEIPT,AMOUNT_RECEIPT,FIXED_OR_FLOATING_PAYMENT,FIXED_RATE_PAYMENT,FLOATING_RATE_INDEX_PAYMENT,FLOATING_RATE_SPREAD_PAYMENT,CURRENCY_CODE_PAYMENT,AMOUNT_PAYMENT
0,S000053068,2026-06-30,0000836906-26-000476,2f5ee059234d2e28f16d0451d9fcb405aab3dad5d92cde...,32866,177102600,Y,2029-06-10,0.00,USD,...,SOFRRATE,0.0,USD,0.0,Fixed,3.9875,None,NaN,USD,0.0
1,S000053068,2026-06-30,0000836906-26-000476,2f5ee059234d2e28f16d0451d9fcb405aab3dad5d92cde...,88959,177110291,Y,2027-04-29,-8.94,USD,...,CPURNSA,0.0,USD,0.0,Fixed,3.4552,None,NaN,USD,0.0
2,S000053068,2026-06-30,0000836906-26-000476,2f5ee059234d2e28f16d0451d9fcb405aab3dad5d92cde...,77600,177112531,Y,2027-04-30,-8.80,USD,...,CPURNSA,0.0,USD,0.0,Fixed,3.5352,None,NaN,USD,0.0
3,S000053068,2026-06-30,0000836906-26-000476,2f5ee059234d2e28f16d0451d9fcb405aab3dad5d92cde...,88393,177100827,Y,2032-04-02,0.00,USD,...,None,NaN,USD,0.0,Floating,NaN,SOFRRATE,0.0,USD,0.0
4,S000053068,2026-06-30,0000836906-26-000476,2f5ee059234d2e28f16d0451d9fcb405aab3dad5d92cde...,22724,177106916,Y,2027-05-08,-7.68,USD,...,CPURNSA,0.0,USD,0.0,Fixed,3.2942,None,NaN,USD,0.0


## 6. Select historical snapshots before joining outcomes
Availability defaults to a lag proxy, not a verified public timestamp. The function filters eligible amendments first and returns age in days. It does not claim a fully point-in-time-certified history.

In [8]:
asof = research.snapshots_asof("2026-03-31", DB)
display(asof[["series_id", "report_date", "filing_date", "available_date_assumed", "availability_basis", "snapshot_age_days"]].head(15))


,series_id,report_date,filing_date,available_date_assumed,availability_basis,snapshot_age_days
0,S000000713,2025-12-31,2026-02-23,2026-03-02,"max(filing_date, report_date+60d)+1d; unverifi...",90.0
1,S000000714,2025-12-31,2026-02-23,2026-03-02,"max(filing_date, report_date+60d)+1d; unverifi...",90.0
2,S000000753,2025-12-31,2026-02-17,2026-03-02,"max(filing_date, report_date+60d)+1d; unverifi...",90.0
3,S000000754,2025-12-31,2026-02-17,2026-03-02,"max(filing_date, report_date+60d)+1d; unverifi...",90.0
4,S000001147,2025-12-31,2026-02-26,2026-03-02,"max(filing_date, report_date+60d)+1d; unverifi...",90.0
5,S000002136,2025-11-30,2026-01-27,2026-01-30,"max(filing_date, report_date+60d)+1d; unverifi...",121.0
6,S000002497,2025-10-31,2025-12-23,2025-12-31,"max(filing_date, report_date+60d)+1d; unverifi...",151.0
7,S000002564,2025-12-31,2026-02-26,2026-03-02,"max(filing_date, report_date+60d)+1d; unverifi...",90.0
8,S000002586,2025-10-31,2025-12-23,2025-12-31,"max(filing_date, report_date+60d)+1d; unverifi...",151.0
9,S000003477,2025-11-30,2026-01-28,2026-01-30,"max(filing_date, report_date+60d)+1d; unverifi...",121.0


## 7. Prepare the external-data crosswalk
Keep ID mappings separate from the raw outputs. Fill from verified sources. Use temporal validity where mappings change. This cell does not overwrite an existing mapping.

In [9]:
crosswalk_file = Path("external_id_crosswalk.csv")
if not crosswalk_file.exists():
    template = coverage[["series_id", "fund_name"]].copy()
    for col in ["class_id", "bloomberg_fund_id", "crsp_fundno", "valid_from", "valid_to", "source"]:
        template[col] = ""
    template.to_csv(crosswalk_file, index=False)
print("Crosswalk:", crosswalk_file)
# Future fee merge: series_id + class_id + effective month after publication/effective-date handling.
# Future factor merge: month_end; retain each factor's own release/publication timing for forecasts.


Crosswalk: external_id_crosswalk.csv


## 8. Optional Parquet export in bounded chunks
Run this only if you want separate Parquet holdings files. Install pyarrow first. SQLite remains the source database. Use a new destination folder to avoid stale files.

In [10]:
EXPORT_PARQUET = False
if EXPORT_PARQUET:
    destination = Path("holdings_parquet")
    destination.mkdir(exist_ok=False)
    for i, chunk in enumerate(pd.read_sql_query("SELECT * FROM holdings_enriched_latest", con, chunksize=100_000)):
        chunk.to_parquet(destination / f"part-{i:05d}.parquet", index=False)


## 9. Review diagnostics
No dates, holdings or returns are forward-filled. Missing periods and outliers need investigation; the pipeline never deletes them automatically.

In [11]:
display(pd.read_sql_query("SELECT * FROM coverage_gaps", con).head(20))
display(pd.read_sql_query("SELECT series_id,report_date,usd_rate_sensitivity_years_signed,risk_review FROM risk_review_flags WHERE risk_review <> 'WITHIN_MAGNITUDE_BOUNDS_SIGN_UNVERIFIED'", con))
con.close()


,series_id,report_date,previous_report_date,gap_days


,series_id,report_date,usd_rate_sensitivity_years_signed,risk_review
0,S000004071,2019-09-30,NaN,MISSING_OR_INCOMPLETE_USD_RISK
1,S000004071,2019-12-31,NaN,MISSING_OR_INCOMPLETE_USD_RISK
2,S000004071,2020-03-31,NaN,MISSING_OR_INCOMPLETE_USD_RISK
3,S000004071,2020-06-30,NaN,MISSING_OR_INCOMPLETE_USD_RISK
4,S000004071,2020-09-30,NaN,MISSING_OR_INCOMPLETE_USD_RISK
5,S000004071,2020-12-31,NaN,MISSING_OR_INCOMPLETE_USD_RISK
6,S000004071,2021-03-31,NaN,MISSING_OR_INCOMPLETE_USD_RISK
7,S000004071,2021-06-30,NaN,MISSING_OR_INCOMPLETE_USD_RISK
8,S000004071,2021-09-30,NaN,MISSING_OR_INCOMPLETE_USD_RISK
9,S000004071,2021-12-31,NaN,MISSING_OR_INCOMPLETE_USD_RISK
